# 02 — Problem Framing

## Why this matters

Before writing an evaluation harness or a model, we need to pin down exactly what the model
predicts, what counts as "correct," and what information it's allowed to see. Get this wrong
and everything downstream is wrong in a way metrics won't catch: a model can score great on
the wrong objective, or score great because it's cheating on data it shouldn't have seen.
This notebook makes those three decisions explicit, using what notebook 1's EDA found.

In [1]:
import pandas as pd

from groovn.data import DATA, load_ratings
from groovn.split import temporal_split

ratings = load_ratings().sort_values("ts").drop_duplicates(subset=["user_id", "item_id"], keep="last")
albums = pd.read_parquet(DATA / "albums.parquet", columns=["item_id", "average_rating", "rating_number"])
len(ratings)

4772071

## Explicit vs. implicit signal

- **Explicit feedback**: the rating value itself (1-5 stars) — a direct, intentional signal
  of preference strength.
- **Implicit feedback**: the fact that an interaction happened at all (a view, a purchase, a
  review) — indirect, unintentional, and much more abundant in real systems (most users never
  rate anything, they just act).

This dataset only has explicit ratings (someone chose to write a review), but notebook 1 found
73.6% of them are 5 stars. That skew means the *value* of the rating carries less information
than the *fact that they rated at all* — reviewing is itself the strong implicit signal
("this album was worth telling people about"), with the star count as a secondary, noisier
refinement.

**Decision:** treat every rating as a positive implicit interaction. Use the rating value as
a *confidence weight* on that interaction (a 5-star review is stronger evidence of fit than a
1-star one — someone still engaged enough to write a review) rather than as a regression
target. This is the standard move for the `implicit`-style ALS used in notebook 5: it expects
a confidence matrix, not a ratings matrix.

## Rating prediction vs. top-K ranking

- **Rating prediction**: predict the exact value a user would give an item; scored with
  RMSE/MAE. Optimizes for getting individual numbers close.
- **Top-K ranking**: predict which items a user is *most likely to want next*, scored with
  ranking metrics (Recall@K, NDCG@K). Optimizes for getting the right items near the top of a
  short list — which is what "recommend 10 albums" actually needs.

A model can win at one and lose at the other. A model that always predicts "4.5" gets a
deceptively good RMSE on this dataset (most true ratings are 4 or 5) while being useless for
ranking — it assigns the same score to every item, so it can't tell a user's favorite genre
from one they'd hate.

In [2]:
baseline_rmse = ((ratings["rating"] - ratings["rating"].mean()) ** 2).mean() ** 0.5
print(f"RMSE of predicting the global mean rating for every (user, item): {baseline_rmse:.3f}")
print("On a 1-5 scale, under 1.0 RMSE looks 'good' while carrying zero personalization signal.")

RMSE of predicting the global mean rating for every (user, item): 1.004
On a 1-5 scale, under 1.0 RMSE looks 'good' while carrying zero personalization signal.


**Decision:** optimize for top-K ranking (Recall@K, NDCG@K, coverage — built in notebook 3),
not rating prediction. It's the metric that actually reflects "did we show the user something
they wanted," and it's robust to the rating skew in a way RMSE isn't.

## Leakage

Leakage means the model sees information at training time that it wouldn't have in
production — usually something from the future relative to the prediction point. Two leakage
traps apply directly here.

**1. Random train/test splits leak the future into the past.** If we split rows randomly, a
user's 2023 ratings can land in train while their 2015 ratings land in test — the model would
be asked to "predict" something that happened chronologically *before* data it was trained on.
Notebook 1 showed review volume is dense enough across 2010-2021 to support a clean temporal
cutoff, so every later notebook splits on a single global timestamp: everything before it is
train, everything after is test.

In [2]:
train, test, cutoff = temporal_split(ratings, test_frac=0.2)
print(f"cutoff: {cutoff}")
print(f"train: {len(train):,} rows up to cutoff, test: {len(test):,} rows after")

cutoff: 2018-04-28 22:48:02.912000
train: 3,817,657 rows up to cutoff, test: 954,414 rows after


**2. Pre-aggregated catalog features can smuggle in the future.** `albums.parquet` ships
`average_rating` and `rating_number` — aggregated over the *entire* dataset history,
including every rating in our test period. Using them as model features would let the model
peek at test-period consensus through the back door. The effect is concrete, not
theoretical — the cell below compares that full-history average against the same average
computed from train-only ratings.

In [3]:
leak_avg = albums.set_index("item_id")["average_rating"]
train_only_avg = train.groupby("item_id")["rating"].mean()

compare = pd.concat([leak_avg.rename("full_history"), train_only_avg.rename("train_only")], axis=1).dropna()
compare["abs_diff"] = (compare["full_history"] - compare["train_only"]).abs()
print(compare["abs_diff"].describe().round(3))
print(f"items where the two averages differ by more than 0.5 stars: {(compare['abs_diff'] > 0.5).mean():.1%}")

count    591431.00
mean          0.38
std           0.51
min           0.00
25%           0.00
50%           0.20
75%           0.50
max           4.00
Name: abs_diff, dtype: float64
items where the two averages differ by more than 0.5 stars: 22.9%


**Reading the result:** for almost a quarter of items, the catalog's `average_rating` differs
from the train-only average by more than half a star — exactly the kind of gap a model could
exploit to look accurate without having learned anything generalizable. **Decision:** any
album-level aggregate feature (`average_rating`, `rating_number`) used by a model from
notebook 4 onward gets recomputed from `train` only, never read from `albums.parquet` directly.

## The task, stated precisely

Putting the three decisions together, every notebook from here on solves the same task:

> Given a user's interaction history up to a cutoff time, and a catalog of albums, rank the
> catalog so that albums the user goes on to interact with after the cutoff appear near the
> top — evaluated only among items present in the training catalog, only for users who have
> at least one training-period interaction (true zero-history cold-start ranking is a
> separate, harder problem noted but out of scope), using Recall@K / NDCG@K / coverage@K as
> the scorecard, never RMSE.

Notebook 3 turns this into runnable code: the exact split, the candidate universe, and the
metric implementations.